# DCAMF-Net 信号分析可视化
跟着一段水声信号走完整个网络，看每步发生了什么。

## 1. 输入信号 — 送进网络的是什么？

In [ ]:
import os, sys, json
import torch, numpy as np, soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import welch, spectrogram

os.environ['OMP_NUM_THREADS'] = '8'
PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'dcamf_net'))

from dcamf_net.model import DCAMFNet
from argparse import Namespace

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
args = Namespace(enc_channels=256, enc_kernel_size=80, enc_stride=40,
                 chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
                 ffn_hidden=512, dw_kernel_size=31, checkpoint=None)

model = DCAMFNet(in_channels=1, enc_channels=256, enc_kernel_size=80, enc_stride=40,
                 chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
                 ffn_hidden=512, dw_kernel_size=31).to(device)

ckpt = torch.load(PROJECT_ROOT / 'experiments' / 'dcamf_net' / 'checkpoints' / 'best_SISNR.pth',
                  map_location=device, weights_only=False)
sd = {k: v for k, v in ckpt.items()
      if not k.endswith('.total_ops') and not k.endswith('.total_params')
      and k not in ('total_ops', 'total_params')}
model.load_state_dict(sd)
model.eval()
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')

# Load a test sample
cf = sorted((PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'clean').glob('*.wav'))[0]
clean, sr = sf.read(str(cf))
noisy, _ = sf.read(str(PROJECT_ROOT / 'data' / 'ShipsEar' / 'test1' / 'noisy' / cf.name))
if clean.ndim > 1: clean = clean.mean(1)
if noisy.ndim > 1: noisy = noisy.mean(1)
L = 48000; clean, noisy = clean[:L], noisy[:L]

# Time + PSD + Spectrogram
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
t = np.arange(L) / sr

axes[0,0].plot(t*1000, noisy, '#E74C3C', lw=0.3); axes[0,0].set_title('Noisy Waveform')
axes[0,0].set_xlabel('Time (ms)'); axes[0,0].set_xlim(0, 500)

f, p = welch(noisy, sr, nperseg=1024)
axes[0,1].semilogy(f, p, '#E74C3C', lw=0.8); axes[0,1].set_title('PSD')
axes[0,1].set_xlabel('Hz'); axes[0,1].set_xlim(0, 4000)

f_s, t_s, Sxx = spectrogram(noisy, sr, nperseg=256, noverlap=200)
axes[1,0].pcolormesh(t_s, f_s, 10*np.log10(Sxx+1e-10), shading='auto', cmap='plasma')
axes[1,0].set_title('Noisy Spectrogram'); axes[1,0].set_ylim(0, 4000); axes[1,0].set_xlabel('Time (s)')

axes[1,1].axis('off')
stats = f'Signal Stats:\nDuration: {L/sr:.1f}s\nMax amp: {abs(noisy).max():.3f}\nRMS: {np.sqrt(np.mean(noisy**2)):.4f}'
axes[1,1].text(0.1, 0.5, stats, fontsize=12, va='center', fontfamily='monospace')
plt.tight_layout(); plt.show()

noisy_t = torch.from_numpy(noisy).float().unsqueeze(0).unsqueeze(0).to(device)


## 2. 编码器 — 256个可学习滤波器组

In [ ]:
# Visualize learned filters
enc_weights = model.encoder.conv.weight.detach().cpu().squeeze(1).numpy()  # (256, 80)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show ALL 256 filters as heatmap
axes[0].imshow(enc_weights, aspect='auto', cmap='RdBu_r', extent=[0, 5, 0, 256])
axes[0].set_title('All 256 Encoder Filters (kernel=80 samples = 5ms)')
axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('Filter #')

# Frequency response of 6 representative filters
from numpy.fft import fft, fftfreq
freq = fftfreq(1024, 1/sr)
for fi, c in zip([0, 20, 60, 100, 160, 220], plt.cm.tab10.colors):
    resp = np.abs(fft(enc_weights[fi], n=1024))
    axes[1].plot(freq[:512], resp[:512], color=c, lw=0.8, label=f'Filter {fi}')
axes[1].set_title('Frequency Response of 6 Filters')
axes[1].set_xlabel('Hz'); axes[1].set_xlim(0, 4000); axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Each filter = 5ms kernel, learned to capture specific temporal patterns.')


## 3. 编码表示 — 信号被分解到256维特征空间

In [ ]:
# Set up hooks to capture all intermediate outputs at once
activations = {}
def hk(name):
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            activations[name] = output.detach().cpu()
        elif isinstance(output, (list, tuple)):
            activations[name] = [o.detach().cpu() if isinstance(o, torch.Tensor) else o for o in output]
    return hook

# Register hooks on encoder and all 10 DCAM blocks + decoder
hooks = []
hooks.append(model.encoder.conv.register_forward_hook(hk('enc_conv')))
hooks.append(model.encoder.prelu.register_forward_hook(hk('enc_prelu')))
hooks.append(model.encoder.register_forward_hook(hk('enc_full')))
for b_idx in range(10):
    blk = model.dcam_blocks[b_idx]
    hooks.append(blk.register_forward_hook(hk(f'b{b_idx}_out')))
    hooks.append(blk.global_cemhsa.pw_conv1.register_forward_hook(hk(f'b{b_idx}_global_glu_in')))
    hooks.append(blk.local_cemhsa.pw_conv1.register_forward_hook(hk(f'b{b_idx}_local_glu_in')))
hooks.append(model.decoder.register_forward_hook(hk('dec_out')))

# Single forward pass — captures everything
with torch.no_grad():
    enhanced = model(noisy_t)
enhanced_np = enhanced.squeeze().cpu().numpy()

# Show encoded features
enc_conv = activations['enc_conv'][0].numpy()      # (256, T_enc)
enc_prelu = activations['enc_prelu'][0].numpy()    # (256, T_enc)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
t_enc = np.arange(enc_conv.shape[1]) * 40 / sr * 1000

# Full 256-channel heatmap (conv output)
im0 = axes[0,0].imshow(enc_conv, aspect='auto', origin='lower', cmap='RdBu_r',
                         extent=[t_enc[0], t_enc[-1], 0, 256])
axes[0,0].set_title('After Conv1d (256 channels)'); axes[0,0].set_xlabel('Time (ms)')
plt.colorbar(im0, ax=axes[0,0])

# Full 256-channel after PReLU
im1 = axes[0,1].imshow(enc_prelu, aspect='auto', origin='lower', cmap='RdBu_r',
                         extent=[t_enc[0], t_enc[-1], 0, 256])
axes[0,1].set_title('After PReLU (negatives preserved)'); axes[0,1].set_xlabel('Time (ms)')
plt.colorbar(im1, ax=axes[0,1])

# Pre vs Post PReLU for first 16 channels
pre16 = enc_conv[:16]; post16 = enc_prelu[:16]
vmax = max(abs(pre16).max(), abs(post16).max())
axes[1,0].imshow(pre16, aspect='auto', origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  extent=[t_enc[0], t_enc[-1], 0, 16])
axes[1,0].set_title('First 16 Ch: Before PReLU'); axes[1,0].set_xlabel('Time (ms)')
axes[1,1].imshow(post16, aspect='auto', origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  extent=[t_enc[0], t_enc[-1], 0, 16])
axes[1,1].set_title('First 16 Ch: After PReLU'); axes[1,1].set_xlabel('Time (ms)')
plt.tight_layout(); plt.show()

print(f'Encoded shape: {enc_conv.shape} (channels x time frames)')
print(f'T_enc = {enc_conv.shape[1]} frames (stride=40, from {L} samples)')
print(f'Segmentation: K=500, H=250, producing ~{(enc_conv.shape[1]-500)//250+1} chunks')


## 4. 双分支注意力 — 局部 vs 全局建模

In [ ]:
# Compare local and global branch outputs across blocks 0, 4, 9
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for row, b_idx in enumerate([0, 4, 9]):
    local_key = f'b{b_idx}_local_glu_in'
    global_key = f'b{b_idx}_global_glu_in'

    local = activations.get(local_key)
    global_ = activations.get(global_key)

    if isinstance(local, torch.Tensor) and len(local.shape) >= 2:
        lf = local[0, :, :min(80, local.shape[-1])].numpy()
        im = axes[row,0].imshow(lf, aspect='auto', cmap='RdBu_r')
        axes[row,0].set_title(f'Block {b_idx} Local Branch (intra-chunk)')
        axes[row,0].set_xlabel('Time in chunk'); axes[row,0].set_ylabel('Feature dim')
        plt.colorbar(im, ax=axes[row,0])

    if isinstance(global_, torch.Tensor) and len(global_.shape) >= 2:
        gf = global_[0, :, :min(50, global_.shape[-1])].numpy()
        im = axes[row,1].imshow(gf, aspect='auto', cmap='RdBu_r')
        axes[row,1].set_title(f'Block {b_idx} Global Branch (cross-chunk)')
        axes[row,1].set_xlabel('Chunk index'); axes[row,1].set_ylabel('Feature dim')
        plt.colorbar(im, ax=axes[row,1])

plt.tight_layout(); plt.show()
print('Local branch: models patterns WITHIN each chunk (high temporal detail).')
print('Global branch: models patterns ACROSS chunks (long-range context).')
print('Shallow blocks (0): rich detail. Deep blocks (9): abstract patterns.')


## 5. GLU 门控 — 信号与门控的可视化

In [ ]:
# Visualize GLU: signal path (H1) vs gate (sigmoid(H2)) for block 0
glu_global = activations.get('b0_global_glu_in')
glu_local = activations.get('b0_local_glu_in')

if glu_global is not None and glu_local is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    for col, (data, name) in enumerate([
        (glu_global, 'Global'), (glu_local, 'Local')
    ]):
        ch = data.shape[1]; half = ch // 2
        sig = data[0, :half, :].numpy()          # H1: signal
        gate = torch.sigmoid(data[0, half:, :]).numpy()  # H2: gate

        im0 = axes[0,col].imshow(sig[:16, :min(80, sig.shape[-1])], aspect='auto', cmap='RdBu_r')
        axes[0,col].set_title(f'{name}: Signal Path H1'); axes[0,col].set_xlabel('Seq')
        plt.colorbar(im0, ax=axes[0,col])

        im1 = axes[1,col].imshow(gate[:16, :min(80, gate.shape[-1])], aspect='auto',
                                  cmap='RdYlGn', vmin=0, vmax=1)
        axes[1,col].set_title(f'{name}: Gate sigmoid(H2)'); axes[1,col].set_xlabel('Seq')
        plt.colorbar(im1, ax=axes[1,col])

    plt.tight_layout(); plt.show()
    print('GLU = H1 * sigmoid(H2).')
    print('H1 learns the signal representation; sigmoid(H2) decides what to keep (green) vs suppress (red).')


## 6. 逐层掩码 — 10个DCAM块各产出什么掩码

In [ ]:
# Extract masks from all 10 blocks (2nd element of block output tuple)
block_masks = {}
for b_idx in range(10):
    out = activations.get(f'b{b_idx}_out')
    if out is not None and isinstance(out, (list, tuple)) and len(out) >= 2:
        m = out[1]
        if isinstance(m, torch.Tensor):
            block_masks[b_idx] = m

if block_masks:
    cols = 5; rows = 2
    fig, axes = plt.subplots(rows, cols, figsize=(20, 7))
    for idx, b_idx in enumerate(sorted(block_masks.keys())):
        ax = axes.flat[idx]
        m = block_masks[b_idx][0, :16, :min(50, block_masks[b_idx].shape[-1])].numpy()
        ax.imshow(m, aspect='auto', cmap='RdBu_r')
        ax.set_title(f'Block {b_idx}'); ax.set_xlabel('T'); ax.set_ylabel('Ch')
    plt.suptitle('Mask from Each DCAM Block (first 16 channels)', fontsize=14)
    plt.tight_layout(); plt.show()

    # Mask energy per block
    fig, ax = plt.subplots(figsize=(10, 4))
    energy = [torch.norm(block_masks[b]).item() for b in sorted(block_masks.keys())]
    ax.bar(sorted(block_masks.keys()), energy, color='#1C7293', edgecolor='black', lw=0.5)
    ax.set_xlabel('Block'); ax.set_ylabel('||Mask||_2')
    ax.set_title('Mask Energy per Block'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()
    print('Shallow masks: sharper, more local detail.')
    print('Deep masks: smoother, capturing global structure.')


## 7. 掩码融合 — 10层掩码如何加权合成

In [ ]:
fw = ckpt.get('mask_fusion_weights')
if fw is not None:
    fw = fw.cpu().numpy() if isinstance(fw, torch.Tensor) else np.array(fw)

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, 10))
    ax.bar(range(1, 11), fw, color=colors, edgecolor='black', lw=0.5)
    for i, w in enumerate(fw):
        ax.text(i+1, w, f'{w:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_xlabel('Layer'); ax.set_ylabel('Fusion Weight')
    ax.set_title('Mask Fusion Weights (Softmax) — learned, fixed after training')
    ax.set_xticks(range(1, 11)); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()
    print('These weights are GLOBAL parameters, learned during training, same for all inputs.')
    print(f'Dominant layer: {np.argmax(fw)+1} (weight={fw.max():.3f})')


## 8. 噪声估计 — 模型认为什么是噪声

In [ ]:
dec_out = activations.get('dec_out')
if isinstance(dec_out, torch.Tensor):
    n_est = dec_out.squeeze().numpy()[:L]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    t = np.arange(L) / sr * 1000

    # Noise waveform
    axes[0,0].fill_between(t[:2000], 0, n_est[:2000], color='#E67E22', alpha=0.4)
    axes[0,0].plot(t[:2000], n_est[:2000], '#E67E22', lw=0.4)
    axes[0,0].set_title('Estimated Noise n(t)'); axes[0,0].set_xlabel('Time (ms)')

    # Noise PSD vs Noisy PSD
    f_n, p_n = welch(noisy, sr, nperseg=1024)
    _, p_ne = welch(n_est, sr, nperseg=1024)
    mask = (f_n >= 0) & (f_n <= 4000)
    axes[0,1].semilogy(f_n[mask], p_n[mask], 'gray', lw=0.5, alpha=0.6, label='Noisy')
    axes[0,1].semilogy(f_n[mask], p_ne[mask], '#E67E22', lw=0.8, label='Estimated noise')
    axes[0,1].set_title('Noise PSD'); axes[0,1].legend(fontsize=8)
    axes[0,1].set_xlabel('Hz')

    # Compare: noisy, noise, clean
    for ax, sig, c, lbl in [(axes[1,0], noisy, '#E74C3C', 'Noisy'),
                              (axes[1,0], n_est, '#E67E22', 'Noise est')]:
        ax.plot(t[:2000], sig[:2000], c, lw=0.3, label=lbl, alpha=0.7)
    axes[1,0].plot(t[:2000], clean[:2000], 'k', lw=0.5, label='Clean')
    axes[1,0].set_title('Waveform Overlay'); axes[1,0].legend(fontsize=7)
    axes[1,0].set_xlabel('Time (ms)')

    # Energy pie
    axes[1,1].pie([np.mean(clean**2), np.mean(n_est**2), np.mean((noisy-clean-n_est)**2)],
                   labels=['Signal', 'Noise removed', 'Residual'],
                   colors=['#27AE60', '#E67E22', '#E74C3C'], autopct='%1.1f%%')
    axes[1,1].set_title('Energy Decomposition')
    plt.tight_layout(); plt.show()

    print(f'Noise power removed: {np.mean(n_est**2):.4f} ({np.mean(n_est**2)/np.mean(noisy**2)*100:.1f}% of input)')


## 9. 输出信号 — 降噪效果对比

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
t = np.arange(L) / sr

# Waveform comparison (first 300ms)
for ax, sig, c, lbl in [(axes[0,0], noisy, '#E74C3C', 'Noisy'),
                          (axes[0,0], enhanced_np, '#1C7293', 'Enhanced'),
                          (axes[0,0], clean, 'k', 'Clean')]:
    ax.plot(t[:5000], sig[:5000], c, lw=0.3, label=lbl, alpha=0.7)
axes[0,0].set_title('Waveform (first 300ms)'); axes[0,0].set_xlabel('Time (s)')
axes[0,0].legend(fontsize=7); axes[0,0].set_xlim(0, 0.3)

# PSD comparison
f_c, p_c = welch(clean, sr, nperseg=1024)
f_n, p_n = welch(noisy, sr, nperseg=1024)
f_e, p_e = welch(enhanced_np, sr, nperseg=1024)
mask_f = (f_c >= 0) & (f_c <= 4000)
axes[0,1].plot(f_c[mask_f]/1000, 10*np.log10(p_c[mask_f]), 'k', lw=1.2, label='Clean')
axes[0,1].plot(f_n[mask_f]/1000, 10*np.log10(p_n[mask_f]), '#E74C3C', lw=0.6, alpha=0.6, label='Noisy')
axes[0,1].plot(f_e[mask_f]/1000, 10*np.log10(p_e[mask_f]), '#1C7293', lw=1.0, label='Enhanced')
axes[0,1].set_title('PSD Comparison'); axes[0,1].set_xlabel('kHz'); axes[0,1].legend(fontsize=7)
axes[0,1].grid(alpha=0.3)

# Spectrogram: Noisy vs Enhanced
f_s, t_s, Sxx_n = spectrogram(noisy, sr, nperseg=256, noverlap=200)
_, _, Sxx_e = spectrogram(enhanced_np, sr, nperseg=256, noverlap=200)
for ax, Sxx, title, cmap in [(axes[1,0], Sxx_n, 'Noisy', 'plasma'),
                               (axes[1,1], Sxx_e, 'Enhanced', 'plasma')]:
    ax.pcolormesh(t_s, f_s, 10*np.log10(Sxx+1e-10), shading='auto', cmap=cmap)
    ax.set_title(title); ax.set_ylim(0, 4000); ax.set_xlabel('Time (s)')
plt.tight_layout(); plt.show()

import IPython.display as ipd
print('Audio:'); ipd.display(ipd.Audio(noisy, rate=sr))
ipd.display(ipd.Audio(enhanced_np, rate=sr))
ipd.display(ipd.Audio(clean, rate=sr))


## 10. 信号流总结

In [ ]:
# Feature energy through the pipeline
stages = ['Input', 'Encoder', 'Block0', 'Block4', 'Block9', 'Decoder', 'Output']
energies = [
    np.sqrt(np.mean(noisy**2)),
    float(torch.norm(activations['enc_full']).item()) / 1000,
    float(torch.norm(activations['b0_out'][0] if isinstance(activations['b0_out'], (list,tuple)) else activations['b0_out']).item()) / 1000,
    float(torch.norm(activations['b4_out'][0] if isinstance(activations['b4_out'], (list,tuple)) else activations['b4_out']).item()) / 1000,
    float(torch.norm(activations['b9_out'][0] if isinstance(activations['b9_out'], (list,tuple)) else activations['b9_out']).item()) / 1000,
    float(torch.norm(activations['dec_out']).item()) / 1000,
    np.sqrt(np.mean(enhanced_np**2)),
]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E74C3C'] + ['#1C7293']*5 + ['#27AE60']
ax.bar(range(7), energies, color=colors, edgecolor='black', lw=0.5)
ax.set_xticks(range(7)); ax.set_xticklabels(stages)
ax.set_ylabel('RMS Energy (scaled)'); ax.set_title('Signal Energy Through the Network')
for i, e in enumerate(energies):
    ax.text(i, e, f'{e:.1f}', ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print('Signal flow: x(t) -> Encoder -> 10 DCAM blocks -> Mask fusion -> Decoder -> n(t) -> s(t)=x(t)-n(t)')
print(f'SI-SNRi improvement: {float(10*np.log10(np.sum(enhanced_np**2)/np.sum((clean-enhanced_np)**2)+1e-8) - 10*np.log10(np.sum(noisy**2)/np.sum((clean-noisy)**2)+1e-8)):.2f} dB')
